> **Research record.** Cleaned copy of `XXX-calculate-WFH/201-validation-geohash-stat_area-home-work.ipynb` (data checks, dead ends and unlabelled plots removed; outputs cleared).
> Builds the monthly geohash layer of detected homes and work locations, aggregates it to statistical areas and joins census attributes. The validation against the census is in `06-validate-home-vs-census-article.ipynb`.
>
> Requires the authors' BigQuery project and the full GPS dataset, which are not public; the notebook is kept as a record of the analysis behind the paper. Run it with its own folder as the working directory and `settings.ini` at the repository root. For a runnable walk-through on synthetic data see `demo/`.

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from geofunctions import utils
import os

In [ ]:
months = utils.return_list_months_ok()

## Layer with home locations by geohash

In [ ]:
full_list_df = []

In [ ]:
months_files = os.listdir(utils.get_path("processed", "dwells", "work"))
months_files = [f for f in months_files if f.endswith(".csv")]

In [ ]:
for monthly_file in sorted(months_files):

    MONTH = monthly_file.split("_")[-1][:-4]
    print(MONTH)
    # Layer with Home location
    home_df = pd.read_csv(
        os.path.join(
            utils.get_path("processed", "dwells", "home"),
            f"home_geohashes_night_shabbat_{MONTH}.csv",
        )
    )
    home_df_geohash = home_df.groupby("geohash")["identifier"].count().reset_index()
    home_df_geohash["geometry"] = home_df_geohash["geohash"].apply(
        utils.geohash_to_polygon
    )
    home_df_geohash = utils.make_gdf(home_df_geohash, crs=4326, geometry="geometry")

    # Layer with work locations in geohash
    work_df = pd.read_csv(
        os.path.join(
            utils.get_path("processed", "dwells", "work"),
            f"users_work_home_{MONTH}.csv",
        ),
        index_col=0,
    )
    work_df_geohash = (
        work_df.groupby("geohash_work")["identifier"].count().reset_index()
    )
    work_df_geohash["geometry"] = work_df_geohash["geohash_work"].apply(
        utils.geohash_to_polygon
    )
    work_df_geohash = work_df_geohash.rename(columns={"geohash_work": "geohash"})
    work_df_geohash = utils.make_gdf(work_df_geohash, crs=4326, geometry="geometry")

    # Merge the two datasets (outer join)
    merged_df = pd.merge(home_df_geohash, work_df_geohash, on="geohash", how="outer")

    # Fill NaN values with 0 for counts
    merged_df["num_homes"] = merged_df["identifier_x"].fillna(0).astype(int)
    merged_df["num_works"] = merged_df["identifier_y"].fillna(0).astype(int)
    merged_df = merged_df.drop(["identifier_x", "identifier_y"], axis=1)

    # Determine type based on prevailing numbers
    merged_df["type"] = np.where(
        merged_df["num_homes"] > merged_df["num_works"], "Home", "Work"
    )
    merged_df["geometry"] = np.where(
        merged_df["geometry_x"].isnull(),
        merged_df["geometry_y"],
        merged_df["geometry_x"],
    )
    merged_df = merged_df.drop(["geometry_x", "geometry_y"], axis=1)
    merged_df["month"] = MONTH
    full_list_df.append(merged_df)

In [ ]:
full_df = pd.concat(full_list_df)

# Explore

In [ ]:
import matplotlib.pyplot as plt

# Grouping and plotting
ax = (
    full_df.groupby("month")[["num_homes", "num_works"]]
    .sum()
    .plot(
        figsize=(10, 6),  # Adjust the figure size
        marker="o",  # Add markers to the lines
        linestyle="-",  # Set line style
        linewidth=2,  # Set line width
    )
)

# Customize the plot
ax.set_title("Monthly Sum of Homes and Works", fontsize=16, fontweight="bold")
ax.set_xlabel("Month", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.legend(title="Category", fontsize=10)
ax.grid(visible=True, linestyle="--", alpha=0.7)  # Add a grid

# Add a red dashed horizontal line at y=5000
plt.axhline(y=5000, color="red", linestyle="--", linewidth=1.5, label="Threshold: 5000")

# Make the x-axis labels more readable if 'month' is not numerical
months = (
    full_df["month"].unique().tolist()
)  # Extract unique months  # Spread ticks evenly
ax.set_xticks(range(len(months)))
ax.set_xticklabels(months, rotation=45, fontsize=10)
# Show the plot
plt.tight_layout()  # Adjust layout to prevent clipping
plt.show()

In [ ]:
full_df["flag_good_month"] = (
    full_df.groupby("month")["num_works"].transform("sum") > 5000
)

In [ ]:
full_df.to_file("../data/to_map/geohash_work_home_stat_monthly_210325.geojson")

# Next steps should be performed in another notebooks Aggregate by stat area and validate

In [ ]:
gdf_census_gush = gpd.read_file(
    utils.get_path("processed", "adm", "geohash_stat_area_pairs.geojson")
)

In [ ]:
full_df_census = full_df.merge(
    gdf_census_gush[["geohash", "YISHUV_STAT_2022"]], on="geohash"
)

In [ ]:
statarea_hw = full_df_census.groupby(["YISHUV_STAT_2022", "month"])[
    ["num_homes", "num_works"]
].sum()

## Add census data

In [ ]:
census_gushdan = gpd.read_file(
    utils.get_path("processed", "census", "census_gushdan_main_features.geojson")
)

In [ ]:
statarea_full = census_gushdan.merge(statarea_hw.reset_index(), on="YISHUV_STAT_2022")